# LAB02｜ADK Agent Skill：把公司 SOP 變成能力
## V2.1 學員教學版｜Google ADK + Groq + Agent Skills + MCP + LINE

> **適合對象：** 不熟 Python、第一次接觸 AI Agent 的學員  
> **操作方式：** 由上到下依序按 ▶ 執行。  
> **重要原則：** 看到「👨‍🎓 學生修改區」才需要動手，其餘程式請先不要修改。


## 🎯 LAB02 學習目標

完成後你應該能回答：

- Skill 和 Prompt 有什麼不同？
- `SKILL.md` 是做什麼的？
- `references/`、`scripts/`、`assets/` 為什麼存在？
- Agent 如何在需要時載入 Skill？

### 本 LAB 架構

```text
Google ADK Agent
      │
      ▼
  SkillToolset
      │
      ▼
quotation-decision/
   ├─ SKILL.md
   ├─ references/
   ├─ scripts/
   └─ assets/
```


In [ ]:
# ============================================================
# STEP 1｜安裝本課需要的套件
# ============================================================
# 這一格的目的：
#   把 Google ADK、MCP、LiteLLM、pandas 安裝到 Colab。
#
# 為什麼需要？
#   Colab 預設沒有這些套件，因此第一次執行一定要安裝。
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。直接按左側 ▶ 執行。
#
# ✅ 成功畫面：
#   執行結束後沒有紅色錯誤訊息即可。
#
# ❗常見錯誤：
#   1. 安裝很久：正常，第一次需要下載套件。
#   2. 出現版本衝突：請重新啟動執行階段後再執行一次。
# ============================================================

!pip -q install "google-adk[mcp]>=2.0.0,<3.0.0" "litellm>=1.84,<2" pandas
print("✅ 套件安裝完成")


In [ ]:
# ============================================================
# STEP 2｜上傳課程資料包
# ============================================================
# 這一格的目的：
#   把「企業模擬資料、Agent Skill、MCP Server」一次載入 Colab。
#
# 你要準備的檔案：
#   quotation_agent_v2_runtime.zip
#
# 👨‍🎓 學生修改區：
#   不用改程式，只要執行後選擇上述 ZIP 檔。
#
# ✅ 成功畫面：
#   看到「✅ 課程資料已載入」
#
# ❗常見錯誤：
#   - 上傳錯檔：重新執行這一格並選正確 ZIP。
#   - 沒看到 skills/ 或 data/：代表 ZIP 沒有成功解壓。
# ============================================================

from google.colab import files
import zipfile, os

if not os.path.exists("skills/quotation-decision/SKILL.md"):
    uploaded = files.upload()
    package = next((n for n in uploaded if n.endswith(".zip")), None)
    if not package:
        raise RuntimeError("❌ 請上傳 quotation_agent_v2_runtime.zip")
    with zipfile.ZipFile(package, "r") as z:
        z.extractall(".")

print("✅ 課程資料已載入")
print("目前重要資料夾：")
print(" - data/")
print(" - skills/quotation-decision/")
print(" - mcp_server/")


In [ ]:
# ============================================================
# STEP 3｜從 Colab Secrets 讀取 Groq API Key
# ============================================================
# 這一格的目的：
#   從 Google Colab 左側的「🔑 Secrets」安全讀取 GROQ_API_KEY。
#
# 👨‍🎓 學生事前準備：
#   1. 打開 Colab 左側選單
#   2. 點「🔑 Secrets」
#   3. 新增名稱：GROQ_API_KEY
#   4. Value 貼上你的 Groq API Key
#   5. 開啟「Notebook access」授權
#
# ⚠️ 不要把 API Key 直接寫在 Notebook 程式碼中。
#
# ✅ 成功畫面：
#   顯示「✅ 已從 Colab Secrets 讀取 GROQ_API_KEY」
#
# ❗常見錯誤：
#   KeyError / Secret not found
#   → 代表 Secrets 裡沒有 GROQ_API_KEY，或沒有開 Notebook access。
# ============================================================

import os
from google.colab import userdata

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")

    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY 為空值")

    os.environ["GROQ_API_KEY"] = GROQ_API_KEY

    # 課堂穩定模型
    GROQ_MODEL = "groq/qwen/qwen3.8-27b"

    print("✅ 已從 Colab Secrets 讀取 GROQ_API_KEY")
    print("✅ Groq 設定完成")
    print("Model =", GROQ_MODEL)
    print("教學穩定模式：reasoning_effort = none")

except Exception as e:
    raise RuntimeError(
        "❌ 無法讀取 Colab Secret：GROQ_API_KEY\n\n"
        "請到 Colab 左側「🔑 Secrets」新增：\n"
        "名稱：GROQ_API_KEY\n"
        "值：你的 Groq API Key\n"
        "並開啟 Notebook access。\n\n"
        f"原始錯誤：{e}"
    )


## STEP 4｜先看看 Skill 資料夾

**不要急著寫程式。**

先看一個真正 Skill 長什麼樣子。


In [ ]:
# ============================================================
# STEP 4｜列出 Skill 資料夾內容
# ============================================================
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。
#
# ✅ 成功畫面：
#   你應該看到 SKILL.md、references、scripts、assets
# ============================================================

from pathlib import Path

skill_root = Path("skills/quotation-decision")

for p in sorted(skill_root.rglob("*")):
    if p.is_file():
        print("📄", p)


## STEP 5｜閱讀 `SKILL.md`

一個 Skill 的核心不是程式，而是「這個能力是什麼、什麼時候用、怎麼做」。


In [ ]:
# ============================================================
# STEP 5｜閱讀 SKILL.md
# ============================================================
#
# 請特別注意最前面的：
#
# ---
# name:
# description:
# ---
#
# 這叫 YAML frontmatter。
#
# name        → Skill 的名稱
# description → 告訴 Agent「什麼情況適合用這個 Skill」
# ============================================================

skill_text = (
    Path("skills/quotation-decision/SKILL.md")
    .read_text(encoding="utf-8")
)

print(skill_text)


## STEP 6｜載入 Skill

這一步的白話：

> 「Google ADK，請把 quotation-decision 這項能力載入。」


In [ ]:
# ============================================================
# STEP 6｜把 Skill 載入 Google ADK
# ============================================================
#
# load_skill_from_dir()
#   → 把整個 Skill 資料夾讀進來
#
# SkillToolset
#   → 把 Skill 變成 Agent 可以使用的能力
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。
# ============================================================

from google.adk.skills import load_skill_from_dir
from google.adk.tools import skill_toolset

quotation_skill = load_skill_from_dir(
    Path("skills/quotation-decision")
)

quotation_skill_toolset = skill_toolset.SkillToolset(
    skills=[quotation_skill]
)

print("✅ quotation-decision Skill 已載入")


## STEP 7｜建立有 Skill 的 Agent

注意：

**現在這個 Agent 知道公司報價流程，但還沒有企業資料 Tools。**

所以它知道「怎麼做」，但還不知道「真實成本是多少」。


In [ ]:
# ============================================================
# STEP 7｜建立 Skill Agent
# ============================================================
#
# 這次 tools 裡放的不是一般 function，
# 而是 SkillToolset。
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。
# ============================================================

from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm

skill_agent = LlmAgent(
    name="quotation_skill_agent",

    model=LiteLlm(
        model=GROQ_MODEL,
        reasoning_effort="none",
    ),

    description="知道公司報價 SOP 與審核規則的助理",

    instruction="""
你是企業業務報價決策助理。

當問題與以下情況有關：
- 是否可以報價
- 建議價格
- 報價風險
- 報價審核
- APPROVE / REVIEW / REJECT

請使用 quotation-decision skill。

請使用繁體中文回答。
""",

    tools=[
        quotation_skill_toolset
    ],
)

print("✅ Skill Agent 建立完成")


In [ ]:
# ============================================================
# 共用工具｜讓我們可以直接「問 Agent 問題」
# ============================================================
# 這一格的目的：
#   Google ADK 的 Agent 需要 Runner + Session 才能真正執行。
#
# 你可以把它想成：
#
#   Agent = 員工
#   Runner = 讓員工開始工作的執行器
#   Session = 這一次對話的工作區
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。
# ============================================================

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
import uuid

APP_NAME = "quotation_training"
USER_ID = "student"

async def ask_agent(agent, text: str):
    session_service = InMemorySessionService()
    session_id = str(uuid.uuid4())

    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id,
    )

    runner = Runner(
        agent=agent,
        app_name=APP_NAME,
        session_service=session_service,
    )

    message = types.Content(
        role="user",
        parts=[types.Part(text=text)]
    )

    final_answer = "(沒有取得最終回答)"

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=message,
    ):
        if event.is_final_response() and event.content and event.content.parts:
            texts = [
                p.text for p in event.content.parts
                if getattr(p, "text", None)
            ]
            if texts:
                final_answer = "\n".join(texts)

    return final_answer

print("✅ Agent 執行工具已準備完成")


## STEP 8｜測試 Skill

👨‍🎓 **學生修改區**

可以嘗試：

- 公司報價什麼情況需要 REVIEW？
- 毛利率低於多少要 REJECT？
- 產能不足時可以直接答應客戶嗎？


In [ ]:
# ============================================================
# 👨‍🎓 學生修改區
# ============================================================

QUESTION = "公司報價時，什麼情況需要 REVIEW？"

# ============================================================
# ⚠️ 以下不需要修改
# ============================================================

print(await ask_agent(skill_agent, QUESTION))


## ✅ LAB02 你已經完成

你現在已經知道：

- [x] Skill 不只是 Prompt
- [x] `SKILL.md` 是核心
- [x] `references/` 放詳細規則
- [x] `scripts/` 放確定性計算
- [x] `assets/` 放範本或資源
- [x] ADK 可以透過 `SkillToolset` 載入 Skill

### 一句話記住

> **Skill = 告訴 Agent「事情應該怎麼做」。**

下一個 LAB：**讓 Agent 透過 MCP 取得企業事實。**
